Since the original "Logistics.DB" file was generated for running queries using duckdb engine, it's now useless for the schema.json generation, since the DuckDB and SQLite are two different database engines. Even though they both use the .db extension sometimes, their internal file formats are not compatible. Because SQLAlchemy (which LangChain uses for the SQL Agent) expects a SQLite database by default, it is looking at your DuckDB file and saying, "I don't recognize this." To keep the project professional and compatible with the LangChain Agent, we will convert those 5 CSV tables into a SQLite database.

### Making of the sqlite DB file:

In [2]:
import pandas as pd
import sqlite3
import os

# 1. Define your file paths
csv_folder = r"B:\3. Prog\2. Projects\7. Logistics and supply chain"
db_path = os.path.join(csv_folder, "logistics_sqlite.db")

# Exact names from your list
csv_files = [
    "dim_drivers.csv",
    "dim_routes.csv",
    "dim_suppliers.csv",
    "dim_vehicles.csv",
    "fact_shipments.csv"
]

# 2. Create a fresh SQLite connection
conn = sqlite3.connect(db_path)

try:
    for file in csv_files:
        table_name = file.replace(".csv", "")
        file_path = os.path.join(csv_folder, file)
        
        if os.path.exists(file_path):
            df = pd.read_csv(file_path)
            df.to_sql(table_name, conn, if_exists='replace', index=False)
            print(f"✅ Table '{table_name}' imported successfully.")
        else:
            print(f"❌ Skipping: {file} not found in folder.")

    print(f"\n🚀 New SQLite DB created at: {db_path}")

finally:
    conn.close()

✅ Table 'dim_drivers' imported successfully.
✅ Table 'dim_routes' imported successfully.
✅ Table 'dim_suppliers' imported successfully.
✅ Table 'dim_vehicles' imported successfully.
✅ Table 'fact_shipments' imported successfully.

🚀 New SQLite DB created at: B:\3. Prog\2. Projects\7. Logistics and supply chain\logistics_sqlite.db


### Making of the schema.json file

In [3]:
import json
import os
from sqlalchemy import create_engine, inspect

# 1. Setup Connection to the NEW SQLite file
db_path = r"B:\3. Prog\2. Projects\7. Logistics and supply chain\logistics_sqlite.db"
engine = create_engine(f'sqlite:///{db_path}')
inspector = inspect(engine)

# 2. Extract Schema Information
db_schema = {}

try:
    tables = inspector.get_table_names()
    for table_name in tables:
        columns = []
        for column in inspector.get_columns(table_name):
            columns.append({
                "name": column['name'],
                "type": str(column['type']),
                "nullable": column['nullable']
            })
        db_schema[table_name] = columns

    # 3. Save to the Metadata folder
    output_path = os.path.join('metadata', 'schema.json')
    with open(output_path, 'w') as f:
        json.dump(db_schema, f, indent=4)

    print(f"✅ schema.json generated successfully with {len(tables)} tables.")
    print(f"📍 Location: {os.path.abspath(output_path)}")

except Exception as e:
    print(f"❌ Failed to generate schema: {e}")

✅ schema.json generated successfully with 5 tables.
📍 Location: b:\3. Prog\2. Projects\7. Logistics and supply chain\metadata\schema.json
